<center><h2><span style="font-weight:bolder; color:darkgoldenrod; font-size:120%">Naive Bayes Project: Diabetes Prediction</span></h2></center>

<center><h2><span style="font-weight:bolder; color:black; font-size:90%">Melissa Jalali Monfared</span></h2></center>

<a id="content"></a>    
<div style="border-radius:20px; padding: 15px; font-size:110%; text-align:left">

<center><h2><span style="font-weight:bolder; color:black; font-size:70%">       Table of Contents:</span></h2></center>

 *  **[- | Introduction](#in)**
 *  **[- | About Dataset](#about)**
 *  **[- | PreProcessing & Visualization](#pre)**
 *  **[- | ML: Naive Bayes](#ml)**

<a id="in"></a>
# <p style="background-color:burlywood;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;">Introduction</p>

### Diabetes, often called "sugar" but not containing sugar itself, is a chronic condition where blood sugar levels (glucose) are consistently higher than normal. It's one of the most common diseases globally and can affect anyone, regardless of age or location.With machine learning models, we can predict diabetes and we must do our best to increase the accuracy of the model and not endanger the health of the patients with incorrect predictions. One of the models that can be used for this prediction is logistic regression.

<a id="about"></a>
# <p style="background-color:burlywood;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;">About Dataset</p>

### This dataset consists of 768 observations & 8 numerical independent variables.
#### Dependent and target variable is OUTCOME. **1** means diabetes test result being positive, **0** means indicates negative.
* **Pregnancies**: Number of Times Being Pregnant
* **Glucose**: Plasma Glucose Concentration (a 2 hours in an oral glucose tolerance test)
* **BloodPressure**: Diastolic Blood Pressure (mm Hg)
* **SkinThickness**: Triceps Skin Fold Thickness (mm)
* **Insulin**: 2-Hour Serum Insulin (mu U/ml)
* **BMI**: Body Mass Index (weight in kg/(height in m)^2)
* **DiabetesPedigreeFunction**: Diabetes Pedigree Function
* **Age**: Age
* **Outcome**: Class variable ( 0 - 1)

<a id="pre"></a>
# <p style="background-color:burlywood;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;">PreProcessing & Visualization</p>

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, cross_validate
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.width', 500)
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot  as plt
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn import preprocessing
from sklearn.naive_bayes import GaussianNB

In [ ]:
data = pd.read_csv("/kaggle/input/diabetes/diabetes.csv")
data = data.copy()

<center><h2><span style="font-weight:bolder; color:black; font-size:80%">Removing Pregnancies Column</span></h2></center>

In [ ]:
df = pd.DataFrame(data.drop('Pregnancies', axis=1))
df

In [ ]:
df.describe(include='all')

### we can see statistical information on the table above

In [ ]:
def check_df(dataframe: object, head: object = 5) -> object:
    print("Shape")
    print(dataframe.shape)
    print("Types")
    print(dataframe.dtypes)
    print("NANs")
    print(dataframe.isnull().sum())
    print("Quantiles")
    print(dataframe.quantile([0, 0.05,0.1, 0.25, 0.50,0.75, 0.90, 0.95, 0.99, 1]).T)
check_df(df)

### there is no NAN data & no object variable type

In [ ]:
df.info()

In [ ]:
def correlated_map(dataframe, plot=False):
    corr = dataframe.corr()
    if plot:
        sns.set(rc={'figure.figsize': (16, 8)})
        sns.heatmap(corr, cmap="BrBG", annot=True, linewidths=.6)
        plt.xticks(rotation=60, size=10)
        plt.yticks(size=10)
        plt.title('Analysis of Correlations', size=14)
        plt.show()
correlated_map(df, plot=True)

### we can see the correlations from above, highest correlations are between Age&Pregnancies, Outcome&Glucose, SkinThickness&Insulin Glucose have the highest correlation with outcome which is our target after that, BMI, Age & Pregnancies have the highest correlation with outcome in comparison with other features

<center><h2><span style="font-weight:bolder; color:black; font-size:80%">Removing Noises</span></h2></center>

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["Glucose"] , df["Outcome"] , color = "sienna")
plt.title ("Relationship between Glucose & Diabetes (the density is visible)" ,fontsize = 20)
plt.xticks (range (0 , 205 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('Glucose', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
df1 = df[df['Glucose']<25] #noises
df1

In [ ]:
df.nsmallest(10, columns='Glucose') #smallest values

In [ ]:
df.drop(df.index[[75, 182, 342, 349, 502]], inplace=True)
df

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["Glucose"] , df["Outcome"] , color = "sienna")
plt.title ("Relationship between Glucose & Diabetes (the density is visible & noises have been removed)" ,fontsize = 20)
plt.xticks (range (0 , 205 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('Glucose', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["BloodPressure"] , df["Outcome"] , color = "sienna")
plt.title ("Relationship between BloodPressure & Diabetes (the density is visible)", fontsize = 20)
plt.xticks (range (0 , 140 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('BloodPressure', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
df2= df[df['BloodPressure']<20]
df2

In [ ]:
print(len(df2))

In [ ]:
df.drop(df.nsmallest(35, columns='BloodPressure').index, inplace=True)

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["BloodPressure"] , df["Outcome"] , color = "sienna")
plt.title ("Relationship between BloodPressure & Diabetes (the density is visible & noises have been removed)" ,fontsize = 20)
plt.xticks (range (0 , 120 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('BloodPressure', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["SkinThickness"] , df["Outcome"] , color = "sienna")
plt.title ("Relationship between SkinThickness & Diabetes (the density is visible)", fontsize = 20)
plt.xticks (range (0 , 120 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('SkinThickness', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
df3 = df[df['SkinThickness']==0]
df3

In [ ]:
df.nsmallest(194, columns='SkinThickness')

In [ ]:
df.drop(df.nsmallest(194, columns='SkinThickness').index, inplace=True)

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["SkinThickness"] , df["Outcome"] , color = "sienna")
plt.title ("Relationship between SkinThickness & Diabetes (the density is visible & noises have been removed)" ,fontsize = 20)
plt.xticks (range (0 , 110 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('SkinThickness', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["Insulin"] , df["Outcome"] , color = "sienna")
plt.title ("Relationship between Insulin & Diabetes (the density is visible)" ,fontsize = 20)
plt.xticks (range (0 , 1000 , 100), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('Insulin', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
df4 = df[df['Insulin']==0]
df4

In [ ]:
df.drop(df[df['Insulin']==0].index, inplace=True)

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["Insulin"] , df["Outcome"] , color = "sienna")
plt.title ("Relationship between Insulin & Diabetes (the density is visible & noises have been removed)" ,fontsize = 20)
plt.xticks (range (0 , 1000 , 100), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('Insulin', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["BMI"] , df["Outcome"] , color = "sienna")
plt.title ("Relationship between BMI & Diabetes (the density is visible)" ,fontsize = 20)
plt.xticks (range (0 , 70 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('BMI', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
df5=df[df['BMI']==0]
df5

In [ ]:
df.drop(df[df['BMI']==0].index, inplace=True)

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["BMI"] , df["Outcome"] , color = "sienna")
plt.title ("Relationship between BMI & Diabetes (the density is visible & noises have been removed)" ,fontsize = 20)
plt.xticks (range (0 , 70 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('BMI', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["Age"] , df["Outcome"] , color = "sienna")
plt.title ("Relationship between Age & Diabetes (the density is visible)" ,fontsize = 20)
plt.xticks (range (0 , 100 , 10), fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('Age', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
plt.figure(figsize = [20, 4] , dpi = 150) 
plt.scatter (df["DiabetesPedigreeFunction"] , df["Outcome"] , color = "sienna")
plt.title ("Relationship between DiabetesPedigreeFunction & Diabetes (the density is visible)" ,fontsize = 20)
plt.xticks (fontsize = 20)
plt.yticks (fontsize = 20)
plt.xlabel ('DiabetesPedigreeFunction', fontsize = 20 )
plt.ylabel ('Diabetes' , fontsize = 20)
plt.grid ()
plt.show ()

In [ ]:
df.describe()

In [ ]:
df.reset_index(drop=True, inplace=True) # reseting index
df

### after removing the detected noises, there is 392 rows left 

<center><h2><span style="font-weight:bolder; color:black; font-size:80%">Visualization</span></h2></center>

<center><h2><span style="font-weight:bolder; color:black; font-size:80%">average of each Feature VS each Outcome</span></h2></center>

In [ ]:
columns1 = ['Glucose','BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
df_mean = df.groupby('Outcome')[columns1].mean()
df_mean.plot(kind='barh', stacked=True, figsize=(15, 10), cmap="BrBG")
plt.xlabel('Average')
plt.title('average of each Feature VS each Outcome')
plt.legend(loc='lower right')
plt.show()

<center><h2><span style="font-weight:bolder; color:black; font-size:80%">influence of each column on Outcome</span></h2></center>

In [ ]:
n = df.groupby('Outcome')[['Glucose','BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']].mean()
n.plot(kind='bar', figsize=(15, 10), cmap='BrBG')
plt.title("influence of each column on Outcome")
plt.xlabel('Outcome')
plt.ylabel('Average')
plt.show()

<center><h2><span style="font-weight:bolder; color:black; font-size:80%">DENSITY PLOTS</span></h2></center>

In [ ]:
columns = ['Glucose','BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
palette ="copper"
for column in columns:
    plt.figure(figsize=(15,2))
    sns.boxplot(x=df[column], palette=palette)
    plt.title(column)
    stats = df[column].describe()
    stats_text = ", ".join([f"{key}: {value:.2f}" for key, value in stats.items()])
    print(f"\n{column} Statistics:\n{stats_text}")
    plt.show()

In [ ]:
columns = ['Glucose','BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
palette ="copper"
for column in columns:
    plt.figure(figsize=(15,4))
    sns.violinplot(x=df[column], palette=palette)
    plt.title(column)
    plt.show()

In [ ]:
plt.subplots(figsize=(20,15))
sns.boxplot(x='Age', y='Glucose', data=df, palette='BrBG')

In [ ]:
plt.subplots(figsize=(20,15))
sns.boxplot(x='Age', y='BloodPressure', data=df, palette='copper')

In [ ]:
plt.subplots(figsize=(20,15))
sns.boxplot(x='Age', y='SkinThickness', data=df, palette='BrBG')

In [ ]:
plt.subplots(figsize=(20,15))
sns.boxplot(x='Age', y='Insulin', data=df, palette='copper')

In [ ]:
plt.subplots(figsize=(20,15))
sns.boxplot(x='Age', y='BMI', data=df, palette='BrBG')

In [ ]:
plt.subplots(figsize=(20,15))
sns.boxplot(x='Age', y='DiabetesPedigreeFunction', data=df, palette='copper')

### the dispersion of Each Feature at each age can be seen in above plots

<center><h2><span style="font-weight:bolder; color:black; font-size:80%">PAIRPLOT</span></h2></center>

In [ ]:
sns.pairplot(data=df, diag_kind='kde', hue='Outcome',palette='copper')
plt.show()

<center><h2><span style="font-weight:bolder; color:black; font-size:80%">HISTOGRAMS</span></h2></center>

### Normal distribution, also known as the Gaussian distribution, is a probability distribution that is symmetric about the mean, showing that data near the mean are more frequent in occurrence than data far from the mean.
### In graphical form, the normal distribution appears as a "bell curve".
### The normal distribution describes a symmetrical plot of data around its mean value, where the width of the curve is defined by the standard deviation. It is visually depicted as the "bell curve."
<img src="https://d2a032ejo53cab.cloudfront.net/Glossary/zl7vYCwx/std3.png" width="400" height="300">

### Skewness measures the degree of symmetry of a distribution. The normal distribution is symmetric and has a skewness of zero. If the distribution of a data set instead has a skewness less than zero, or negative skewness (left-skewness), then the left tail of the distribution is longer than the right tail; positive skewness (right-skewness) implies that the right tail of the distribution is longer than the left.
<img src="https://www.biologyforlife.com/uploads/2/2/3/9/22392738/c101b0da6ea1a0dab31f80d9963b0368_orig.png" width="800" height="600">

### The normal distribution follows the following formula. Note that only the values of the mean (μ ) and standard deviation (σ) are necessary
<img src="https://www.investopedia.com/thmb/lFaG1vgFO0XgA_Xzfw3yPLjG2Iw=/750x0/filters:no_upscale():max_bytes(150000):strip_icc():format(webp)/Clipboard01-fdb217713438416cadafc48a1e4e5ee4.jpg" width="400" height="300">

####    x = value of the variable or data being examined and f(x) the probability function
####    μ = the mean
####    σ = the standard deviation

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sns.distplot(df['Glucose'].dropna(),kde=True,color='sienna')

In [ ]:
plt.figure(figsize=(40,20),dpi=90)
ax=sns.countplot(x='Glucose',data=df, palette='copper')
plt.xticks(rotation=90, fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel('Glucose',fontsize=20)
plt.ylabel('Diabetes',fontsize=20)
plt.title('Count of Glucose',fontsize=30)
plt.grid()

### positive skewness (right-skewness) is visible for Glucose

In [ ]:
glucose_bins=pd.cut(df["Glucose"],bins=[40,90,130,200],labels=["40-90","90-130","130-200"])
plt.figure(figsize=(20,10))
sns.countplot(x=glucose_bins,data=df,hue="Outcome",palette='copper')

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sns.distplot(df['BloodPressure'].dropna(),kde=True,color='sienna')

In [ ]:
plt.figure(figsize=(40,20),dpi=90)
ax=sns.countplot(x='BloodPressure',data=df, palette='copper')
plt.xticks(rotation=90, fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel('BloodPressure',fontsize=20)
plt.ylabel('Diabetes',fontsize=20)
plt.title('Count of BloodPressure',fontsize=30)
plt.grid()

### negative skewness (left-skewness) is visible for BloodPressure

In [ ]:
plt.figure(figsize=(20,10))
sns.countplot(x="BloodPressure",data=df,hue="Outcome",palette="copper")

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sns.distplot(df['SkinThickness'].dropna(),kde=True,color='sienna')

In [ ]:
plt.figure(figsize=(40,20),dpi=90)
ax=sns.countplot(x='SkinThickness',data=df, palette='copper')
plt.xticks(rotation=90, fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel('SkinThickness',fontsize=20)
plt.ylabel('Diabetes',fontsize=20)
plt.title('Count of SkinThickness',fontsize=30)
plt.grid()

### positive skewness (right-skewness) is visible for SkinThickness

In [ ]:
plt.figure(figsize=(20,10))
sns.countplot(x="SkinThickness",data=df,hue="Outcome",palette="copper")

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sns.distplot(df["BMI"].dropna(),kde=True,color='sienna')

In [ ]:
plt.figure(figsize=(40,10),dpi=80)
ax=sns.countplot(x='BMI',data=df, palette='copper')
plt.xticks(rotation=90, fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel('BMI',fontsize=20)
plt.ylabel('Diabetes',fontsize=20)
plt.title('Count of BMI',fontsize=30)
plt.grid()

### positive skewness (right-skewness) is visible for BMI

In [ ]:
BMI_bins=pd.cut(df["BMI"],bins=[15,20,25,30,35,40,45,50,55,60,65,70],labels=["15-20","20-25","25-30","30-35","35-40","40-45","45-50","50-55","55-60","60-65","65-70"])
plt.figure(figsize=(20,10))
sns.countplot(x=BMI_bins,data=df,hue="Outcome",palette='copper')

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sns.distplot(df["Insulin"].dropna(),kde=True,color='sienna')

In [ ]:
plt.figure(figsize=(40,10),dpi=80)
ax=sns.countplot(x='Insulin',data=df, palette='copper')
plt.xticks(rotation=90, fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel('Insulin',fontsize=20)
plt.ylabel('Diabetes',fontsize=20)
plt.title('Count of Insulin',fontsize=30)
plt.grid()

### positive skewness (right-skewness) is visible for Insulin

In [ ]:
BMI_bins=pd.cut(df["Insulin"],bins=[0,50,100,150,200,300,400,500,600,700,800,900,1000],labels=["0-50","50-100","100-150","150-200","200-300","300-400","400-500","500-600","600-700","700-800","800-900","900-1000"])
plt.figure(figsize=(20,10))
sns.countplot(x=BMI_bins,data=df,hue="Outcome",palette='copper')

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sns.distplot(df["Age"].dropna(),kde=True,color='sienna')

In [ ]:
plt.figure(figsize=(40,10),dpi=80)
ax=sns.countplot(x='Age',data=df, palette='copper')
plt.xticks(rotation=90, fontsize=20)
plt.yticks(fontsize=20)
plt.xlabel('Age',fontsize=20)
plt.ylabel('Diabetes',fontsize=20)
plt.title('Count of Age',fontsize=30)
plt.grid()

### positive skewness (right-skewness) is visible for Age

In [ ]:
plt.figure(figsize=(20,10))
sns.countplot(x="Age",data=df,hue="Outcome",palette="copper")

In [ ]:
import warnings
warnings.filterwarnings("ignore")
sns.distplot(df['DiabetesPedigreeFunction'].dropna(),kde=True,color='sienna')

In [ ]:
DiabetesPedigreeFunction_bins=pd.cut(df["DiabetesPedigreeFunction"],bins=[0,0.3,0.6,0.9,1.2,1.5,1.8,2.1,2.4,2.7,3],labels=["0-0.3","0.3-0.6","0.6-0.9","0.9-1.2","1.2-1.5","1.5-1.8","1.8-2.1","2.1-2.4","2.4-2.7","2.7-3"])
plt.figure(figsize=(20,10))
sns.countplot(x=DiabetesPedigreeFunction_bins,data=df,hue="Outcome",palette='copper')

### positive skewness (right-skewness) is visible for DiabetesPedigreeFunction

<center><h2><span style="font-weight:bolder; color:black; font-size:80%">3D PLOTS</span></h2></center>

In [ ]:
fig = plt.figure(figsize=(14, 10),dpi=100)
ax = fig.add_subplot(111,projection='3d')
p1 = ax.scatter(df['BMI'], df['Age'], df['Glucose'],c=df['Outcome'],cmap='BrBG')
fig.colorbar(p1, shrink=0.3,label='Outcome',anchor=(3,1))
ax.set_xlabel("BMI")
ax.set_ylabel("Age")
ax.set_zlabel("Glucose")
ax.set_title("Correlation Between Glucose & Age & BMI",fontdict={'fontsize': 12})
ax.patch.set_facecolor("white")
fig.show()

In [ ]:
fig = plt.figure(figsize=(14, 10),dpi=100)
ax = fig.add_subplot(111,projection='3d')
p1 = ax.scatter(df['BloodPressure'], df['SkinThickness'], df['Glucose'],c=df['Outcome'],cmap='BrBG')
fig.colorbar(p1, shrink=0.3,label='Outcome',anchor=(3,1))
ax.set_xlabel("BloodPressure")
ax.set_ylabel("SkinThickness")
ax.set_zlabel("Insulin")
ax.set_title("Correlation Between Insulin & SkinThickness & BloodPressure",fontdict={'fontsize': 12})
ax.patch.set_facecolor("white")
fig.show()

<a id="ml"></a>
# <p style="background-color:burlywood;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;">ML: Naive Bayes</p>

In [ ]:
# normalizing
scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))
dff = scaler.fit_transform(df)
dff = pd.DataFrame(dff, columns=['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI','DiabetesPedigreeFunction', 'Age', 'Outcome'])

In [ ]:
# creating X and Y (separating independent variables from dependent variables)
X = pd.DataFrame(dff.drop('Outcome', axis=1))
Y = dff['Outcome'].values.reshape(-1, 1)

In [ ]:
# creating train & test data (test size=0.2)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size= 0.2, random_state=0)

In [ ]:
# creating model
model = GaussianNB()
model.fit(X_train, Y_train.ravel())

In [ ]:
# prediction
y_pred = model.predict(X_test)

In [ ]:
from sklearn import metrics
print("\33[43m Accuracy Is:", metrics.accuracy_score(Y_test, y_pred))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
model.classes_

In [ ]:
confusion_matrix(Y, model.predict(X))

In [ ]:
# confusion matrix heatmap
plt.figure(figsize=(15, 8))
sns.set(font_scale=1.2)
sns.heatmap(confusion_matrix(Y, model.predict(X)) , annot=True, fmt="d", cmap="BrBG", cbar=False,
            xticklabels=['Predict 0s', 'Predict 1s'], yticklabels=['Actual 0s', 'Actual 1s'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### 226 people did not have diabetes and it was predicted that they did not have diabetes
### 79 people has diabetes and it was predicted that they have diabetes
### 36 people did not have diabetes and it was predicted that they have diabetes
<center><h2><span style="font-weight:bolder; color:Red; font-size:100%">51 people has diabetes and it was predicted that they did not have diabetes</span></h2></center>

In [ ]:
print(classification_report(Y, model.predict(X)))

In [ ]:
# entering new numbers to peredict
newf = model.predict([[60, 32, 43, 93, 29.7, 0.371, 60]])
newf

In [ ]:
# improving
kfold = KFold(5)
print(cross_val_score(model, X, Y.ravel(), cv=kfold, n_jobs=1))

In [ ]:
dff1 = dff[dff.index < 78]
dff2 = dff[dff.index > 157]
dff3 = pd.concat([dff1, dff2], ignore_index=True)
dff3.reset_index(drop=True, inplace=True)
dff3

In [ ]:
391 / 5 

In [ ]:
78.2 * 2

In [ ]:
# normalizing
scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))
normal = scaler.fit_transform(dff3)
df_normal = pd.DataFrame(normal, columns=['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI','DiabetesPedigreeFunction', 'Age', 'Outcome'])

In [ ]:
# creating X and Y (separating independent variables from dependent variables)
x = pd.DataFrame(df_normal.drop('Outcome', axis=1))
y = df_normal.Outcome.values.reshape(-1, 1)

In [ ]:
# creating train & test data (test size=0.2)
x_train, x_test, y_train, y_test = train_test_split(x, y , test_size=0.2, random_state=0)

In [ ]:
# creating model
model1 = GaussianNB()
model1.fit(x_train, y_train.ravel())

In [ ]:
# prediction
y_pred2 = model1.predict(x_test)

In [ ]:
print('\33[43m Accuracy :', metrics.accuracy_score(y_test, y_pred2))

In [ ]:
kfold = KFold(5)
print(cross_val_score(model, x, y.ravel(), cv=kfold, n_jobs=1))

In [ ]:
print(confusion_matrix(y, model1.predict(x)))

In [ ]:
# confusion matrix heatmap
plt.figure(figsize=(15, 8))
sns.set(font_scale=1.2)
sns.heatmap(confusion_matrix(y, model1.predict(x)) , annot=True, fmt="d", cmap="BrBG", cbar=False,
            xticklabels=['Predict 0s', 'Predict 1s'], yticklabels=['Actual 0s', 'Actual 1s'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### 184 people did not have diabetes and it was predicted that they did not have diabetes
### 65 people has diabetes and it was predicted that they have diabetes
### 32 people did not have diabetes and it was predicted that they have diabetes
<center><h2><span style="font-weight:bolder; color:Red; font-size:100%">31 people has diabetes and it was predicted that they did not have diabetes</span></h2></center>

## It can be seen that the number of destructive and dangerous forecasts has decreased by "20" (which was "51" before improving), so with the changes, we had a more accurate forecast and model has improved.

In [ ]:
print(classification_report(y, model1.predict(x)))

In [ ]:
# peredicting 
newfeature = model1.predict([[60, 32, 43, 93, 29.7, 0.371, 60]])
newfeature